# Elementary Cellular Automata
### Lesson 4, Section 3

In this notebook we will:
1. Understand **1D cellular automata** and the Wolfram numbering system
2. Build a **rule lookup table** from a rule number
3. Implement a **single ECA step**
4. Generate **space-time diagrams**
5. Explore all **256 rules** and Wolfram's four classes
6. Study famous rules: **Rule 30** (chaos), **Rule 90** (fractal), **Rule 110** (Turing complete)

---

## 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

---
## 1 · What Is an Elementary Cellular Automaton?

An **Elementary Cellular Automaton (ECA)** is the simplest class of CA:

- **1D** grid — a row of cells, each either 0 (white) or 1 (black).
- Each cell's next state depends on **itself and its two immediate neighbours** — a 3-cell neighbourhood.
- There are $2^3 = 8$ possible neighbourhood patterns.
- Each pattern maps to 0 or 1, giving $2^8 = 256$ possible rules.

### The Wolfram numbering system

We list all 8 patterns in order (111, 110, 101, 100, 011, 010, 001, 000) and read the outputs as a binary number.

For example, **Rule 110** has outputs `0,1,1,0,1,1,1,0`:

| Pattern | 111 | 110 | 101 | 100 | 011 | 010 | 001 | 000 |
|---------|-----|-----|-----|-----|-----|-----|-----|-----|
| Output  |  0  |  1  |  1  |  0  |  1  |  1  |  1  |  0  |

Binary: $01101110_2 = 110_{10}$

---
## 2 · Step 1: Building the Lookup Table

We convert a rule number (0–255) into a dictionary mapping each 3-cell pattern to its output.

In [ ]:
def rule_to_table(rule_number):
    """Convert a Wolfram rule number to a lookup table.
    
    Parameters
    ----------
    rule_number : int
        Integer 0–255 identifying the ECA rule.
    
    Returns
    -------
    dict : mapping (left, center, right) -> output
    """
    table = {}
    for i in range(8):
        # Extract the 3-bit pattern: i = 4*left + 2*center + right
        left   = (i >> 2) & 1
        center = (i >> 1) & 1
        right  = i & 1
        # The i-th bit of rule_number gives the output
        output = (rule_number >> i) & 1
        table[(left, center, right)] = output
    return table

### Test: Inspect Rule 110

In [ ]:
table_110 = rule_to_table(110)

print('Rule 110 lookup table:')
print(f'{"Pattern":>10} → Output')
print('-' * 22)
for i in range(7, -1, -1):
    pattern = ((i >> 2) & 1, (i >> 1) & 1, i & 1)
    print(f'  {pattern[0]}{pattern[1]}{pattern[2]}       → {table_110[pattern]}')

# Verify: reading outputs from top to bottom gives binary 01101110 = 110
binary_str = ''.join(str(table_110[((i>>2)&1, (i>>1)&1, i&1)]) for i in range(7, -1, -1))
print(f'\nBinary: {binary_str} = {int(binary_str, 2)} (should be 110)')

### Visualise the lookup table

In [ ]:
def plot_rule_table(rule_number):
    """Visualise the 8 pattern-output pairs of an ECA rule."""
    table = rule_to_table(rule_number)
    fig, axes = plt.subplots(1, 8, figsize=(14, 2))
    for idx in range(7, -1, -1):
        ax = axes[7 - idx]
        pattern = ((idx >> 2) & 1, (idx >> 1) & 1, idx & 1)
        output = table[pattern]
        # Draw 3 input cells on top row
        for j, v in enumerate(pattern):
            rect = plt.Rectangle((j, 1), 1, 1,
                                 facecolor='black' if v else 'white',
                                 edgecolor='gray', lw=2)
            ax.add_patch(rect)
        # Draw output cell below centre
        rect = plt.Rectangle((1, 0), 1, 1,
                             facecolor='black' if output else 'white',
                             edgecolor='gray', lw=2)
        ax.add_patch(rect)
        ax.set_xlim(-0.1, 3.1)
        ax.set_ylim(-0.3, 2.3)
        ax.set_aspect('equal')
        ax.axis('off')
    fig.suptitle(f'Rule {rule_number} — Lookup Table', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_rule_table(110)

---
## 3 · Step 2: Applying One ECA Step

Given a row of cells, produce the next row by looking up each cell's neighbourhood.

In [ ]:
def eca_step(row, table):
    """Apply one step of an Elementary Cellular Automaton.
    
    Parameters
    ----------
    row : np.ndarray (1D)
        Current row of cells (0s and 1s).
    table : dict
        Lookup table from rule_to_table.
    
    Returns
    -------
    np.ndarray : the next row
    """
    n = len(row)
    new_row = np.zeros(n, dtype=int)
    for i in range(n):
        left   = row[(i - 1) % n]  # wrap around (periodic boundary)
        center = row[i]
        right  = row[(i + 1) % n]
        new_row[i] = table[(left, center, right)]
    return new_row

### Test: One step of Rule 110

In [ ]:
row = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0])
table = rule_to_table(110)

print('Initial row:', row)
for step in range(1, 5):
    row = eca_step(row, table)
    print(f'Step {step}:     ', row)

---
## 4 · Step 3: Generating a Space-Time Diagram

We run many steps, stacking each row to form a 2D image where time flows downward.

In [ ]:
def run_eca(rule_number, width=201, steps=100, initial=None):
    """Generate the full space-time diagram for an ECA rule.
    
    Parameters
    ----------
    rule_number : int
        Wolfram rule number (0–255).
    width : int
        Number of cells in the row.
    steps : int
        Number of time steps to simulate.
    initial : np.ndarray or None
        Initial row. If None, a single 1 in the centre.
    
    Returns
    -------
    np.ndarray : 2D array (steps × width)
    """
    table = rule_to_table(rule_number)
    grid = np.zeros((steps, width), dtype=int)
    if initial is not None:
        grid[0] = initial
    else:
        grid[0, width // 2] = 1  # single cell in the centre
    for t in range(1, steps):
        grid[t] = eca_step(grid[t - 1], table)
    return grid


def plot_eca(grid, rule_number=None, ax=None):
    """Display a space-time diagram."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))
    ax.imshow(grid, cmap='binary', interpolation='nearest', aspect='auto')
    if rule_number is not None:
        ax.set_title(f'Rule {rule_number}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Cell')
    ax.set_ylabel('Time step')
    return ax

In [ ]:
# Generate and display Rule 110
grid_110 = run_eca(110, width=201, steps=100)
plot_eca(grid_110, rule_number=110)
plt.tight_layout()
plt.show()
print('Each row is one time step. Time flows from top to bottom.')

---
## 5 · Gallery of Rules

Let's compare several well-known rules:

In [ ]:
rules = [30, 90, 110, 184, 54, 150]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, rn in zip(axes.flat, rules):
    g = run_eca(rn, width=201, steps=100)
    plot_eca(g, rule_number=rn, ax=ax)
fig.suptitle('Elementary Cellular Automata — Gallery', fontweight='bold', fontsize=15)
plt.tight_layout()
plt.show()

---
## 6 · Wolfram's Four Classes

Stephen Wolfram classified all 256 rules into four classes:

| Class | Behaviour | Examples |
|-------|-----------|----------|
| I | Converges to **uniform** state | Rule 0, 32, 160 |
| II | Converges to **periodic** structures | Rule 4, 108, 232 |
| III | **Chaotic**, aperiodic patterns | Rule 30, 45, 73 |
| IV | **Complex** localised structures | Rule 110, 54 |

In [ ]:
class_examples = [
    (0,   'Class I — Uniform'),
    (4,   'Class II — Periodic'),
    (30,  'Class III — Chaotic'),
    (110, 'Class IV — Complex'),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (rn, title) in zip(axes, class_examples):
    g = run_eca(rn, width=201, steps=100)
    ax.imshow(g, cmap='binary', interpolation='nearest', aspect='auto')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.axis('off')
fig.suptitle("Wolfram's Four Classes", fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

---
## 7 · Deep Dive: Rule 30 — Chaos

Rule 30 is remarkable: starting from a single cell it produces **chaotic, aperiodic** patterns.

The central column passes statistical tests for randomness — Wolfram used it as a random number generator in *Mathematica*.

In [ ]:
grid_30 = run_eca(30, width=401, steps=200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Space-time diagram
ax1.imshow(grid_30, cmap='binary', interpolation='nearest', aspect='auto')
ax1.set_title('Rule 30 — Space-Time Diagram', fontweight='bold')
ax1.set_xlabel('Cell')
ax1.set_ylabel('Time')

# Central column — is it random?
central_col = grid_30[:, 200]
ax2.step(range(len(central_col)), central_col, where='mid', color='black', lw=0.5)
ax2.set_title('Central Column Values', fontweight='bold')
ax2.set_xlabel('Time step')
ax2.set_ylabel('Value (0 or 1)')
ax2.set_yticks([0, 1])

# Count 0s and 1s
n_ones = np.sum(central_col)
n_zeros = len(central_col) - n_ones
ax2.text(0.95, 0.5, f'0s: {n_zeros}\n1s: {n_ones}',
         transform=ax2.transAxes, ha='right', fontsize=12,
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.show()
print(f'Ratio of 1s: {n_ones/len(central_col):.3f} (close to 0.5 indicates randomness)')

---
## 8 · Deep Dive: Rule 90 — Sierpinski Triangle

Rule 90 is equivalent to XOR of the two neighbours. It produces a perfect **Sierpinski triangle** — a fractal.

In [ ]:
grid_90 = run_eca(90, width=401, steps=200)

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(grid_90, cmap='binary', interpolation='nearest', aspect='auto')
ax.set_title('Rule 90 — Sierpinski Triangle', fontweight='bold', fontsize=14)
ax.set_xlabel('Cell')
ax.set_ylabel('Time step')
plt.tight_layout()
plt.show()
print('Rule 90 = XOR of neighbours. The result is a fractal with self-similar structure at all scales.')

---
## 9 · Deep Dive: Rule 110 — Turing Complete

Rule 110 produces complex, interacting structures (gliders, backgrounds). It was proven **Turing complete** by Matthew Cook in 2004 — meaning it can simulate any computation. See Wikipida page https://en.wikipedia.org/wiki/Rule_110 for details

In [ ]:
grid_110_big = run_eca(110, width=601, steps=300)

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(grid_110_big, cmap='binary', interpolation='nearest', aspect='auto')
ax.set_title('Rule 110 — Complex Structures', fontweight='bold', fontsize=14)
ax.set_xlabel('Cell')
ax.set_ylabel('Time step')
plt.tight_layout()
plt.show()
print('Notice the triangular "gliders" moving left and the regular background pattern.')

---
## 10 · Random Initial Conditions

What happens if we start from a **random** row instead of a single cell?

In [ ]:
np.random.seed(42)
random_init = np.random.randint(0, 2, size=201)

rules_to_try = [30, 90, 110, 184]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, rn in zip(axes, rules_to_try):
    g = run_eca(rn, width=201, steps=100, initial=random_init)
    ax.imshow(g, cmap='binary', interpolation='nearest', aspect='auto')
    ax.set_title(f'Rule {rn}', fontweight='bold')
    ax.axis('off')

fig.suptitle('Random Initial Conditions', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print('Rule 184 is interesting: it models simple traffic flow (1=car, 0=empty space).')

---
## 11 · Explore All 256 Rules

Generate a poster of all rules:

In [ ]:
fig, axes = plt.subplots(16, 16, figsize=(20, 20))

for rule_num in range(256):
    row_idx = rule_num // 16
    col_idx = rule_num % 16
    ax = axes[row_idx, col_idx]
    g = run_eca(rule_num, width=61, steps=30)
    ax.imshow(g, cmap='binary', interpolation='nearest', aspect='auto')
    ax.set_title(str(rule_num), fontsize=6, pad=1)
    ax.axis('off')

fig.suptitle('All 256 Elementary Cellular Automata', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

---
## 12 · What to Try Next

1. **Rule 184 as traffic**: Interpret 1 as a car moving right. Start with a random density of cars. Observe how "traffic jams" form and propagate.
2. **Sensitivity**: For Rule 30, run two simulations with initial rows that differ by a single bit. After how many steps do the patterns diverge completely?
3. **Totalistic rules**: Instead of looking at exact (L, C, R) patterns, use only the **sum** L+C+R. How many totalistic rules exist? What patterns do they produce?
4. **Two-dimensional ECA**: Extend the ECA idea to 2D (outer totalistic rules on a 2D grid). How does the Wolfram numbering generalise?
5. **Compression test**: For Rule 30, extract the central column and try to compress it (e.g., with gzip). Compare with Rule 90. Which is more compressible?

---

## 13 · Mini-Projects

### Project A: Rule Classification
Write code that automatically classifies each of the 256 rules into Wolfram's four classes. Possible heuristics: (a) does the grid become uniform? (Class I), (b) does it become periodic? (Class II), (c) is the density of 1s stable but seemingly random? (Class III), (d) are there localised structures? (Class IV). Compare your classification with Wolfram's.

### Project B: Rule 184 Traffic Model
Implement Rule 184 as a traffic model. Start with a random density $\rho$ of cars. Measure the **flow** (number of cars that move per step) as a function of $\rho$. Plot the **fundamental diagram** of traffic: flow vs. density. Find the critical density where traffic jams form.

### Project C: Cryptographic Properties of Rule 30
Generate 10,000 bits from the central column of Rule 30. Run statistical tests: (a) frequency of 0s and 1s, (b) runs test (consecutive same-valued bits), (c) autocorrelation. Compare with Python's `random` module. Is Rule 30 a good pseudo-random number generator?